# Faktafrågor

## 1. Vad menas med *curse of dimensionality*
> Curse of dimensionality innebär att mer dimensioner eller egenskaper i ett dataset kan göra datan mer extrem. Varje gång en till dimension ökar datasetets volym exponentiellt, det blir svårare och vid en viss punkt omöjligt att mäta "distans". Det gör också datan väldigt svår att förstå, människan kan intuitivt förstå upp till 3 dimensioner och över det går det inte att visualisera. 

## 2. Vad är dimensionsreducering och varför görs det?

> Dimensionsreducering innebär att man tar bort en dimension för att göra datan snabbare att träna på och enklare att visualisera. Man kan göra det med "PCA". Det innebär dock att viss information försvinner. 

## 3. Förklara översiktligt hur *PCA* fungerar. Använd figur 5.4 på sidan 224 i din förklaring.

> PCA gör utifrån dimensionerna i datasetet nya dimensioner som behåller mest information. De rangordnas och man väljer ut de som har högst varians.

# Resonemangsfrågor

## 5. Stina påstår att man i maskininlärning alltid vill ha modeller som genomför så bra prediktioner som möjligt. Kalle påstår att det inte riktigt stämmer eftersom tid också är en viktig aspekt. Både för själva modellträningen och för själva prediktionerna. Vad säger du?
> Stina har rätt att prediktionsprestanda är det centrala målet, men Kalle har en poäng att det sällan är det enda målet, i praktiken måste man ofta avväga bättre prediktioner mot kostnaden i tränings och prediktionstid, särskilt om modellen ska användas i realtid eller med begränsade resurser. Det handlar alltså inte om att Stina har fel, hon har bara inte hela sanningen.

## 6. Efter att vi genomfört en *PCA*, vad händer med tolkningen av variablerna?
> Variablerna går inte längre att tolka, det går inte att säga att en viss variabel är t.ex. vikt. 

# Koduppgifter

## 8. Förklara vad nedanstående kod gör.
```python
    import numpy as np
    from sklearn.decomposition import PCA
    # Creating a dataset with 3 features/columns
    X = np.random.rand(1000, 3)
    print(X[0:5])
    # Reducing the data to 2 dimensions
    pca = PCA(n_components=2)
    X2D = pca.fit_transform(X)
    print(X2D[0:5])
    # "Recreating" the data to 3 dimensions
    X3D_inv = pca.inverse_transform(X2D)
    # Not exactly equal since some information was lost in the transformation
    print(np.allclose(X3D_inv, X))
```

> 1. X instantieras som ett slumpmässigt dataset på 1000 rader och 3 kolumner
> 2. de första 5 raderna skriv ut
> 3. pca instantieras och vi vill ha de 2 bästa dimensionerna
> 4. pca anpassar nya komponenter med högst varians och projicerar sen datan med 3 dimensioner till 2
> 5. de första 5 raderna som nu är bara i 2 dimensioner skrivs nu ut igen
> 6. pca.inverse_transform försöker ta tillbaka datasetet till 3 dimensioner
> 7. sedan jämförs hur lik det nya datasetet är det gamla

## 9. Genomför en *PCA* på “car_price_dataset.csv” från kapitel 3 innan du modellerar det med ML. Hur påverkas resultatet?

In [3]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [4]:
data = pd.read_csv('../data/car_price_dataset.csv', sep=';')
features = data.drop(columns='Price')
target = data['Price']

In [5]:
encoded_features = pd.get_dummies(features, drop_first=True, dtype=float)
train_features, test_features, train_target, test_target = train_test_split(
    encoded_features, target, test_size=0.2, random_state=1337
)
scaler = StandardScaler()
train_features_scaled = scaler.fit_transform(train_features)
test_features_scaled = scaler.transform(test_features)


In [6]:
baseline_model = LinearRegression()
baseline_model.fit(train_features_scaled, train_target)
baseline_predictions = baseline_model.predict(test_features_scaled)

In [7]:
pca = PCA(n_components=0.95)
train_features_pca = pca.fit_transform(train_features_scaled)
test_features_pca = pca.transform(test_features_scaled)
pca_model = LinearRegression()
pca_model.fit(train_features_pca, train_target)
pca_predictions = pca_model.predict(test_features_pca)

In [8]:
print(f'Without PCA: RMSE = {mean_squared_error(test_target, baseline_predictions) ** 0.5:.2f}, R² = {r2_score(test_target, baseline_predictions):.3f}')
print(f'With PCA: RMSE = {mean_squared_error(test_target, pca_predictions) ** 0.5:.2f}, R² = {r2_score(test_target, pca_predictions):.3f}')
print(f'PCA components: {pca.n_components_}')

Without PCA: RMSE = 96.44, R² = 0.999
With PCA: RMSE = 979.48, R² = 0.904
PCA components: 35


PCA minskar antalet variabler, men resultatet blir normalt något sämre eftersom information försvinner.